# Part2 Cnn Images

Notebook version of `part2_cnn_images.py`. Run the cells from top to bottom.

In [ ]:
import pandas as pd

import torch

from sklearn.datasets import load_digits

from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

from sklearn.model_selection import train_test_split

from torch import nn

from torch.utils.data import DataLoader, TensorDataset

from utils import FIGURES_DIR, METRICS_DIR, get_device, plot_confusion_matrix, plot_history, save_json


In [ ]:
def corr2d(x, kernel):
    h, w = kernel.shape
    output = torch.zeros((x.shape[0] - h + 1, x.shape[1] - w + 1))
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            output[i, j] = (x[i:i + h, j:j + w] * kernel).sum()
    return output


In [ ]:
def pool2d(x, pool_size=(2, 2), mode="max"):
    h, w = pool_size
    output = torch.zeros((x.shape[0] // h, x.shape[1] // w))
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            window = x[i * h:(i + 1) * h, j * w:(j + 1) * w]
            output[i, j] = window.max() if mode == "max" else window.mean()
    return output


In [ ]:
class ImageMLP(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, filters=16, padding=1, stride=1, pooling="max", use_1x1=False, num_classes=10):
        super().__init__()
        pool = nn.MaxPool2d(2) if pooling == "max" else nn.AvgPool2d(2)
        layers = [
            nn.Conv2d(1, filters, kernel_size=3, padding=padding, stride=stride),
            nn.ReLU(),
            pool,
        ]
        if use_1x1:
            layers.extend([nn.Conv2d(filters, filters, kernel_size=1), nn.ReLU()])
        self.features = nn.Sequential(*layers)
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 8, 8)
            flattened = self.features(dummy).view(1, -1).shape[1]
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:
def load_image_data(batch_size=64):
    digits = load_digits()
    x = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
    y = torch.tensor(digits.target, dtype=torch.long)
    x_train, x_temp, y_train, y_temp = train_test_split(
        x, y, test_size=0.30, random_state=42, stratify=y
    )
    x_val, x_test, y_val, y_test = train_test_split(
        x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )
    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size)
    return train_loader, val_loader, test_loader, x_test[:1]


In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=12, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": []}
    model.to(device)

    for _ in range(epochs):
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x_batch.size(0)
        history["train_loss"].append(train_loss / len(train_loader.dataset))

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(x_batch), y_batch).item() * x_batch.size(0)
        history["val_loss"].append(val_loss / len(val_loader.dataset))
    return history


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    predictions = []
    targets = []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            pred = model(x_batch.to(device)).argmax(dim=1).cpu()
            predictions.extend(pred.tolist())
            targets.extend(y_batch.tolist())
    precision, recall, f1, _ = precision_recall_fscore_support(
        targets, predictions, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(targets, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": confusion_matrix(targets, predictions),
    }


In [ ]:
def save_feature_maps(model, sample, device):
    import matplotlib.pyplot as plt

    model.eval()
    with torch.no_grad():
        maps = model.features[1](model.features[0](sample.to(device))).cpu()[0]
    count = min(8, maps.shape[0])
    plt.figure(figsize=(8, 3))
    for idx in range(count):
        plt.subplot(2, 4, idx + 1)
        plt.imshow(maps[idx], cmap="viridis")
        plt.axis("off")
        plt.title(f"filtre {idx}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "part2_feature_maps.png", dpi=160)
    plt.close()


In [ ]:
def manual_operation_checks():
    x = torch.tensor([[0., 1., 2.], [3., 4., 5.], [6., 7., 8.]])
    kernel = torch.tensor([[0., 1.], [2., 3.]])
    manual_conv = corr2d(x, kernel)
    torch_conv = nn.functional.conv2d(x.view(1, 1, 3, 3), kernel.view(1, 1, 2, 2)).view(2, 2)
    return {
        "manual_corr2d": manual_conv.tolist(),
        "torch_conv2d": torch_conv.tolist(),
        "max_pool": pool2d(x, mode="max").tolist(),
        "avg_pool": pool2d(x, mode="avg").tolist(),
    }


In [ ]:
def run_part2():
    device = get_device()
    train_loader, val_loader, test_loader, sample = load_image_data()

    configs = [
        {"name": "mlp_baseline", "model": ImageMLP()},
        {"name": "cnn_padding1_stride1_max_16", "model": SmallCNN(filters=16, padding=1, stride=1, pooling="max")},
        {"name": "cnn_padding0_stride1_max_16", "model": SmallCNN(filters=16, padding=0, stride=1, pooling="max")},
        {"name": "cnn_padding1_stride1_avg_16", "model": SmallCNN(filters=16, padding=1, stride=1, pooling="avg")},
        {"name": "cnn_padding1_stride1_max_32", "model": SmallCNN(filters=32, padding=1, stride=1, pooling="max")},
        {"name": "cnn_padding1_stride1_max_16_conv1x1", "model": SmallCNN(filters=16, padding=1, stride=1, pooling="max", use_1x1=True)},
    ]

    rows = []
    best_cnn = None
    best_f1 = -1.0
    for item in configs:
        history = train_model(item["model"], train_loader, val_loader, device)
        metrics = evaluate(item["model"], test_loader, device)
        rows.append({
            "model": item["name"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        })
        if "cnn" in item["name"] and metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_cnn = item["model"]
            plot_history(history, "Partie II - apprentissage CNN", FIGURES_DIR / "part2_cnn_history.png")
            plot_confusion_matrix(
                metrics["confusion_matrix"],
                [str(i) for i in range(10)],
                "Partie II - matrice de confusion CNN",
                FIGURES_DIR / "part2_cnn_confusion.png",
            )

    if best_cnn is not None:
        save_feature_maps(best_cnn, sample, device)

    pd.DataFrame(rows).to_csv(METRICS_DIR / "part2_cnn_results.csv", index=False)
    save_json(manual_operation_checks(), METRICS_DIR / "part2_manual_operations.json")
    print(pd.DataFrame(rows))


In [ ]:
run_part2()
